# Build your own Mixture-of-Experts in 15 minutes

[MixtureKit](https://github.com/MBZUAI-Paris/MixtureKit) composes a Mixture-of-Experts from existing Hugging Face models, trains only its routers, and shows which expert every token uses.

This notebook runs on a free Colab T4 GPU (*Runtime → Change runtime type → T4 GPU*):

1. merge two Qwen3-0.6B models (the base model and the instruct model) into a BTX Mixture-of-Experts,
2. look at its routing, random before training,
3. train only the routers on math problems (a few minutes),
4. see the experts specialize,
5. optionally upload it to the Hub with a generated model card.

In [ ]:
%pip install -q "mixturekit[train,vis] @ git+https://github.com/MBZUAI-Paris/MixtureKit"

## Settings

In [ ]:
import torch

EXPERTS = [
    {"expert_name": "Qwen3-0.6B-Base", "model_id": "Qwen/Qwen3-0.6B-Base"},
    {"expert_name": "Qwen3-0.6B (instruct)", "model_id": "Qwen/Qwen3-0.6B"},
]
MODEL_DIR = "qwen3-2x0.6b-btx"
TRAIN_STEPS = 60          # router training steps
TRAIN_SAMPLES = 400       # math problems used for training
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FP16 = DEVICE == "cuda"     # the T4 has no bf16
print(DEVICE)

## 1. Merge the experts

BTX keeps the feed-forward projections of every expert, each behind a router that picks the top-2 experts for every token; the other weights (attention, embeddings) are averaged. The merge reads the experts tensor by tensor, so it needs little memory.

In [ ]:
from MixtureKit import build_moe

config = {
    "moe_method": "btx",
    "model_type": "qwen3btx",
    "experts": EXPERTS,
    "num_experts_per_tok": 2,
    "router_layers": ["mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"],
    "router_layers_index": [],   # all blocks
    "alpha": 0.01,               # load balancing weight
}
build_moe(config, torch_dtype=torch.float16 if FP16 else torch.float32, output_dir=MODEL_DIR)

## 2. Routing before training

Every token is coloured by the expert its routers prefer, averaged over all layers (hover a token for the weights). The routers are untrained: the choice is random.

In [ ]:
from MixtureKit.routing import load_model, trace_routing

MATH = "Natalia sold 48 clips in April and half as many in May. How many clips did she sell?"
CHAT = "Hey! Could you suggest a nice name for my new cat?"

model, tokenizer = load_model(MODEL_DIR, device=DEVICE)
trace_routing(model, tokenizer, MATH)

## 3. Train the routers

Only the routers are trained (the experts stay frozen). The data: grade-school math problems, formatted as chat.

In [ ]:
import yaml
from datasets import load_dataset

math = load_dataset("openai/gsm8k", "main", split=f"train[:{TRAIN_SAMPLES}]")
math = math.map(lambda row: {"messages": [
    {"role": "user", "content": row["question"]},
    {"role": "assistant", "content": row["answer"]},
]}, remove_columns=math.column_names)
math.save_to_disk("math_data")

training = {
    "model_id": MODEL_DIR,
    "dataset_path": "math_data",
    "output_dir": MODEL_DIR + "-trained",
    "max_steps": TRAIN_STEPS,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 2,
    "learning_rate": 1e-3,
    "lr_scheduler_type": "constant",
    "max_length": 384,
    "gradient_checkpointing": True,
    "fp16": FP16,
    "use_cpu": DEVICE == "cpu",
    "logging_steps": 10,
    "save_strategy": "no",
    "report_to": "none",
}
with open("train.yaml", "w") as f:
    yaml.safe_dump(training, f)

del model
torch.cuda.empty_cache() if DEVICE == "cuda" else None
!mixturekit train train.yaml

## 4. Routing after training

The trained routers send math and chat text to different experts. Compare the two views, and the share of the tokens of every expert in the legend.

In [ ]:
model, tokenizer = load_model(MODEL_DIR + "-trained", device=DEVICE)
display(trace_routing(model, tokenizer, MATH))
display(trace_routing(model, tokenizer, CHAT))

The interactive app shows the per-layer details: `mixturekit visualize qwen3-2x0.6b-btx-trained` (on your machine, or in a Hugging Face Space).

## 5. Upload it to the Hub (optional)

The model card lists the experts, the method, how to load the model (MixtureKit is not needed) and the configuration to rebuild it.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# from MixtureKit.hub import push_to_hub
# push_to_hub(MODEL_DIR + "-trained", "your-name/qwen3-2x0.6b-btx")

---

Next: your own experts (any fine-tunes of one base model), BTS (`"moe_method": "bts"`) and Mixtral-style MoEs (`"traditional"`). See the [README](https://github.com/MBZUAI-Paris/MixtureKit) and cite the [paper](https://arxiv.org/abs/2512.12121) if MixtureKit helps your work.